In [ ]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)

In [ ]:
# Cargar datos

data = pd.read_csv("../data/postings.csv")


In [ ]:
data.head()


In [ ]:
# Revisar si el sueldo tiene valores nulos
filtered_data = data[data["normalized_salary"].isna() == False]

In [ ]:
filtered_data["description"] = np.where(filtered_data["description"].isna(), "no description", filtered_data["description"])
filtered_data["skills_desc"] = np.where(filtered_data["skills_desc"].isna(), "no skills desc", filtered_data["skills_desc"])
filtered_data["title"] = np.where(filtered_data["title"].isna(), "no title", filtered_data["title"])
filtered_data["total_text"] =  filtered_data["description"] + " " + filtered_data["skills_desc"] + " " + filtered_data["title"]
filtered_data[["total_text", "normalized_salary"]].head()



In [ ]:
from sklearn.model_selection import train_test_split

X = filtered_data["total_text"]
y = filtered_data["normalized_salary"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# Procesamiento de texto

import nltk

# Limpieza del texto

import re

def clean_text(text):
    text = str(text)
    # Convertir a minúsculas
    text = text.lower()
    # Eliminar caracteres especiales y números
    text = re.sub(r'[^a-z0-9\s]', '', text)
    # Eliminar espacios extra
    text = re.sub(r'\s+', ' ', text).strip()
    return text

X_train_cleaned = X_train.apply(clean_text)
X_test_cleaned = X_test.apply(clean_text)



In [ ]:
# Tokenización

from nltk.tokenize import word_tokenize

X_train_tokenized = X_train_cleaned.apply(word_tokenize)
X_test_tokenized = X_test_cleaned.apply(word_tokenize)

In [ ]:
# Remover stopwords

from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def remove_stopwords(tokens):
    return [word for word in tokens if word not in stop_words]

X_train_no_stopwords = X_train_tokenized.apply(remove_stopwords)
X_test_no_stopwords = X_test_tokenized.apply(remove_stopwords)


In [ ]:
# Lematizar

from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

def lemmatize_tokens(tokens):
    return [lemmatizer.lemmatize(word) for word in tokens]

X_train_lemmatized = X_train_no_stopwords.apply(lemmatize_tokens)
X_test_lemmatized = X_test_no_stopwords.apply(lemmatize_tokens)

In [ ]:
# Unir tokens de nuevo en texto

def tokens_to_text(tokens):
    return ' '.join(tokens)

X_train_final = X_train_lemmatized.apply(tokens_to_text)
X_test_final = X_test_lemmatized.apply(tokens_to_text)


In [ ]:
# Vectorizar

from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=500)
X_train_vectorized = vectorizer.fit_transform(X_train_final)
X_test_vectorized = vectorizer.transform(X_test_final)



In [ ]:
# Entrenar modelo de regresión

from catboost import CatBoostRegressor

modelo = CatBoostRegressor(iterations=3000, learning_rate=0.01, depth=6)
modelo.fit(X_train_vectorized, y_train, verbose=100)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error
preds = modelo.predict(X_test_vectorized)

mean_absolute_error(y_test, preds)
mean_absolute_percentage_error(y_test, preds)